---
title: "矩阵补全"
date: "2026-03-20"
excerpt: "推荐算法其实和潜在因子模型用的是一套思路，矩阵补全。"
tags: ["矩阵补全", "潜在因子模型", "推荐算法"]
category: ["经济学", "技术"]
language: "zh"
cover: "/images/covers/矩阵补全.png"
---

最近收到了一份非常好玩的coding test，要求复现一篇做可解释性推荐算法的论文。我之前从来没有了解过推荐算法相关的内容，这一次算是在干中学。非常有意思的是，推荐算法其实和潜在因子模型用的是一套思路，矩阵补全。

## Q1：什么样的矩阵可以补全？

从线性代数的角度看，一个矩阵能否被较好地补全，关键不在于“缺了多少值”，而在于它满不满足“信息完全”（低秩）这一假设。

设矩阵 $M\in\mathbb R^{n\times T}$，其秩 $\mathrm{rank}(M)=r$ 表示矩阵中线性独立的行或列最多有多少个。若 $r$ 很小，就意味着这个矩阵的很多行和列并不是彼此独立变化的，而是可以由少数几个基础模式线性生成。换句话说，矩阵中的有效信息维度并不高，存在较强的“冗余结构”。

例如，当 $\mathrm{rank}(M)=r$ 时，我们可以把它写成

$$
M=UV^\prime,
$$

其中 $U\in\mathbb R^{n\times r}$、$V\in\mathbb R^{T\times r}$。这说明，矩阵中看似很多的元素，其实是由少数几个潜在维度共同决定的。如果 $r$ 远小于 $n$ 和 $T$，那么只要我们已经观测到的元素足够多、分布也足够分散，就有可能从这些局部信息中反推出整个矩阵。

相反，如果一个矩阵接近满秩，意味着几乎每一行、每一列都携带新的独立信息，那么缺失一个元素就几乎等于真的丢掉了一块无法由其他位置推回来的信息。在这种情况下，矩阵补全通常是非常困难、甚至不可能的。

因此，矩阵补全最适合处理的，并不是任意缺失矩阵，而是那些虽然表面上维度很高、但本质上由少数潜在因子驱动的低秩矩阵。后面无论是推荐系统中的用户-物品评分矩阵，还是因果推断中的潜在结果矩阵，本质上都在利用这一点。

In [1]:
import numpy as np

# 例子：一个明显低秩的矩阵
u = np.array([[1], [2], [3]])
v = np.array([[2, 4, 6, 8]])
low_rank_matrix = u @ v

# 对比：一个一般的矩阵
full_rank_matrix = np.array([
    [1, 2, 3, 1],
    [0, 1, 4, 2],
    [2, 0, 1, 3]
])

print("低秩矩阵：")
print(low_rank_matrix)
print("rank =", np.linalg.matrix_rank(low_rank_matrix))
print()
print("一般矩阵：")
print(full_rank_matrix)
print("rank =", np.linalg.matrix_rank(full_rank_matrix))
print()
print("低秩矩阵的第 2 行正好等于第 1 行的 2 倍，第 3 行等于第 1 行的 3 倍。")
print("这就是冗余：少量信息就能决定整张矩阵。")

低秩矩阵：
[[ 2  4  6  8]
 [ 4  8 12 16]
 [ 6 12 18 24]]
rank = 1

一般矩阵：
[[1 2 3 1]
 [0 1 4 2]
 [2 0 1 3]]
rank = 3

低秩矩阵的第 2 行正好等于第 1 行的 2 倍，第 3 行等于第 1 行的 3 倍。
这就是冗余：少量信息就能决定整张矩阵。


## Q2:用什么方法进行矩阵补全？

矩阵补全的目标不是随便找一个矩阵去拟合观测值，而是在所有与已观测数据一致的候选矩阵中，找出那个秩尽可能小的矩阵。设真实矩阵为 $X\in\mathbb R^{n\times T}$，被观测到的位置集合记为 $\Omega$，我们观察到的是 $\{X_{it}:(i,t)\in\Omega\}$。那么最直接的表述是

$$
\min_{\hat X}\ \mathrm{rank}(\hat X)
\quad \text{s.t.} \quad \hat X_{it}=X_{it},\ (i,t)\in\Omega.
$$

这个问题的含义非常直观：在所有能够精确匹配已观测元素的矩阵里，我们偏好那个结构最简单、维度最低的解。因为如果真实矩阵确实是低秩的，那么缺失值并不是任意的，而是受到整体低维结构约束的。

不过，直接最小化秩函数是一个非凸、组合意义上很困难的问题。 Candes 等人在 2008 年论文《Exact Matrix Completion via Convex Optimization》中提出：可以用核范数（nuclear norm）作为秩函数的凸替代，从而把原问题放松为一个凸优化问题：

$$
\min_{\hat X}\ \lVert \hat X \rVert_*
\quad \text{s.t.} \quad \hat X_{it}=X_{it},\ (i,t)\in\Omega,
$$

其中 $\lVert \hat X \rVert_*$ 表示矩阵的核范数，也就是所有奇异值之和。之所以这样做，是因为核范数在矩阵情形下扮演的角色，类似于向量稀疏问题中的 $\ell_1$ 范数：它鼓励解具有更低的有效维度，同时又保留了凸优化带来的可计算性。

在满足一定条件时，比如真实矩阵足够低秩、采样位置足够随机、且矩阵满足一定的 incoherence 条件，上述核范数最小化可以精确恢复原始低秩矩阵。这也是为什么矩阵补全不是简单的插值问题，而是一个有明确识别条件和优化理论支撑的问题。

当然，在实际应用中，人们也常把低秩矩阵写成 $\hat X=UV^\prime$ 的因子分解形式来做数值计算；但从理论表述上说，核心问题首先是“找最小秩解”，而核范数最小化则是这一问题最经典的凸松弛形式。

In [2]:
import numpy as np

# 例子：先造一个低秩矩阵
u = np.array([[1], [2], [3]])
v = np.array([[2, 4, 6, 8]])
X_true = u @ v

# 假装我们只观察到一部分元素
X_obs = X_true.astype(float).copy()
X_obs[0, 2] = np.nan
X_obs[2, 1] = np.nan

print("真实矩阵：")
print(X_true)
print()
print("只观察到一部分后的矩阵：")
print(X_obs)
print()

# 因为这里知道它是 rank=1，所以可以写成 X_hat = u_hat v_hat'
# 先用第一列恢复 u_hat，再用第一行恢复 v_hat
u_hat = X_obs[:, [0]] / X_obs[0, 0]
v_hat = X_obs[[0], :]
X_hat = u_hat @ v_hat

print("利用低秩结构补全后：")
print(X_hat)
print()
print("被遮掉的位置被补回来了：")
print("(1,3) 位置 =", X_hat[0, 2])
print("(3,2) 位置 =", X_hat[2, 1])

真实矩阵：
[[ 2  4  6  8]
 [ 4  8 12 16]
 [ 6 12 18 24]]

只观察到一部分后的矩阵：
[[ 2.  4. nan  8.]
 [ 4.  8. 12. 16.]
 [ 6. nan 18. 24.]]

利用低秩结构补全后：
[[ 2.  4. nan  8.]
 [ 4.  8. nan 16.]
 [ 6. 12. nan 24.]]

被遮掉的位置被补回来了：
(1,3) 位置 = nan
(3,2) 位置 = 12.0


## Q3:潜在因子模型是怎么矩阵补全的？

潜在因子模型的核心思想，是把潜在结果框架改写成一个“部分单元缺失”的矩阵恢复问题，目标是估计处理组在处理之后的反事实结果 $Y_{it}(0)$。![数据缺失问题](/images/posts/数据缺失.png)
先写清楚潜在结果。对于个体 $i$ 在时期 $t$，记 $Y_{it}(1)$ 为接受处理的结果，$Y_{it}(0)$ 为未接受处理的结果，处理指示变量为 $W_{it}\in\{0,1\}$，则观测结果满足

$$
Y_{it}=W_{it}Y_{it}(1)+(1-W_{it})Y_{it}(0).
$$

我们真正关心的处理效应是

$$
\tau_{it}=Y_{it}(1)-Y_{it}(0).
$$

一旦某个处理组个体在处理后满足 $W_{it}=1$，我们只能观察到 $Y_{it}(1)$，而对应的反事实 $Y_{it}(0)$ 无法被直接观察（有点像薛定谔的猫，在同一时间你无法同时观测到一个变量的两个状态）。也就是说，在“个体 \times 时间”的结果矩阵中，处理组处理后的那些单元，其未处理结果 $Y_{it}(0)$ 恰恰就是缺失值。

颜色最深的格子可以理解为处理组在处理后的观测结果，次深的是处理组处理前的结果，最浅的是对照组各时期的结果。矩阵补全要做的事情，就是利用那些已经观测到的未处理结果，去推断被遮住的 $Y_{it}(0)$。

关键假设是：未处理状态下的结果矩阵并不是完全任意的，而是可以被一个低维潜在因子结构较好地刻画。例如，Xu (2017) 使用的潜在因子模型可写为

$$
Y_{it}(0)=\alpha_i+\delta_t+\lambda_i^\prime f_t+\varepsilon_{it},
$$

其中，$\alpha_i$ 表示个体固定效应，$\delta_t$ 表示时间固定效应，$f_t$ 是随时间变化的共同冲击，$\lambda_i$ 则描述个体对这些共同冲击的载荷。$\lambda_i^\prime f_t$ 这一项意味着，不同个体即使面对同一个宏观冲击，反应强度也可以不同，因此它比简单的双向固定效应或平行趋势假设更灵活。

如果满足严格的平行趋势，我们确实可以直接用 DID 的差分思路识别反事实；但当不同个体存在异质趋势、共同因子和不同因子载荷时，平行趋势通常不再成立。这时，研究者就转而利用上式的低秩结构，在所有未受处理的观测单元上估计 $\alpha_i,\delta_t,\lambda_i,f_t$，再把估计得到的结构外推到缺失位置，从而补全处理后处理组的 $Y_{it}(0)$。

因此，潜在因子模型做矩阵补全的逻辑可以概括为：不是直接假设“处理组和对照组趋势平行”，而是假设未处理结果矩阵背后存在一个可以被少数潜在因子解释的结构；只要这个结构能被处理前和对照组的数据较好识别，我们就能对缺失的反事实单元进行预测，并进一步计算处理效应。更完整的识别条件与证明可以参看 Xu (2017) 关于 generalized synthetic control 的论文。

## Q4:在推荐算法中是怎么矩阵补全的（以可解释mg-gat为例）？

Q2 里提到的矩阵补全，核心思想是把一个部分观测到的评分矩阵写成低秩结构，例如 $\hat X = UV^\prime$，然后利用已知元素去恢复缺失值。这个思路在推荐系统里当然仍然成立，但相当粗糙：真实的矩阵补全问题是否满足“完全信息”假设？就算满足，可解释性呢？因此，在现代推荐算法里，矩阵补全通常不再被理解为一个纯粹的低秩恢复问题，而是被扩展为一个“结合特征和图结构的评分预测问题”。

以论文中的 MG-GAT 为例，它本质上仍然是在补全一个用户--商家评分矩阵。设 $X_{ij}$ 表示用户 $i$ 对商家 $j$ 的评分，已观测位置来自 Yelp 中真实发生过的评分行为，未观测位置则对应需要预测的缺失项。但 MG-GAT 不再把 $U$ 和 $V$ 简单视为自由可调的低维矩阵，而是试图让它们由更有经济含义和行为含义的信息共同生成。具体来说，$U$ 用户侧不仅有历史评分，还有用户的辅助特征和好友网络；$V$商家侧不仅有历史评分，还有类别、营业时间、属性、地理位置以及商家之间的多种关系图。

论文中的第一步，是先把用户和商家的辅助特征映射到一个统一的表示空间（更改一下矩阵的形状以便训练）。设 $S_{u,i}$ 和 $S_{b,j}$ 分别表示用户 $i$ 和商家 $j$ 的原始辅助信息，则有

$$
H^{(1)}_{u,i}=W_u^{(1)}S_{u,i}, \qquad
H^{(1)}_{b,j}=W_b^{(1)}S_{b,j}.
$$

接下来，MG-GAT 不再假设所有邻居都同等重要，而是在用户图和商家图上学习邻居权重。对一个焦点节点而言，不同邻居对评分预测任务的相关性并不一样：有些好友的消费偏好更接近当前用户，有些商家虽然地理位置相近，但在类别和客群上并不相似。图注意力机制要做的，就是根据节点表示和图关系，给每个邻居分配一个归一化权重 $\alpha$，这些权重构成论文中所说的 neighbor importance graph（NIG）。然后，模型对邻居表示做加权聚合：

$$
\tilde h_i=\sum_{k\in\mathcal N(i)} \alpha_{k\to i} h_k.
$$

这一步和经典矩阵补全最大的不同在于：缺失评分的预测不再只依赖于某个静态的低秩分解，而是依赖于节点在图中的局部结构，以及“哪些邻居为当前预测提供了真正相关的信息”。

在完成邻居聚合之后，模型再把聚合结果与原始辅助特征进一步融合，形成最终的用户表示 $U_i$ 和商家表示 $B_j$。最后，评分预测仍然会落回一个类似矩阵分解的形式，例如

$$
\hat X_{ij}=U_i B_j^\prime + b_i + b_j + \mu,
$$

或者再经过一个映射函数限制在 1 到 5 星之间。也就是说，从最后的输出形式上看，它依然保留了“用户向量与商家向量交互生成评分”的结构；但从这些向量是如何得到的来看，它已经不再是简单的低秩矩阵分解，而是一个由辅助特征、社交网络和商家多图共同塑造的深度表示学习过程。

简而言之，推荐算法中的矩阵补全并不是抛弃了传统矩阵补全，而是把它推广了。经典方法强调“评分矩阵是低秩的，所以可以恢复缺失值”；MG-GAT 则进一步强调“评分矩阵中的缺失值还受到用户特征、商家特征以及图结构的共同约束，因此可以通过图注意力网络学习出更有信息量、也更可解释的 $U$ 和 $V$”。从这个意义上说，MG-GAT 仍然是在做矩阵补全，只不过它做的是一种“图增强的、可解释的矩阵补全”。